In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict ,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [ ]:
load_dotenv()

model = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
class SentimentSchema(BaseModel):
    sentiment : Literal['positive','negative'] = Field(description = 'Sentiment of the review')



In [ ]:
structured_model = model.with_structured_output(SentimentSchema)

prompt = 'What is the sentiment of the following review - the software too good'

structured_model.invoke(prompt).sentiment

In [ ]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal['positive','negative']
    diagnosis: dict
    response: str

In [ ]:
def find_sentiment(state: ReviewState):
    review = state["review"]

    prompt = f"""
    Classify the sentiment of this customer review.

    Review: {review}

    Return exactly one word:
    positive or negative.

    If the review expresses dissatisfaction, complaints,
    or problems, classify it as negative.
    Otherwise, classify it as positive.
    """

    result = model.invoke(prompt).content.strip().lower()

    # Normalize the LLM output
    sentiment = "negative" if "negative" in result else "positive"

    return {"sentiment": sentiment}

In [ ]:
def run_diagnosis(state: ReviewState):
    review = state["review"]

    prompt = f"""
    Analyze this negative customer review.

    Review: {review}

    Identify:
    1. The main problem
    2. The likely cause, if evident
    3. What action could resolve the issue

    Do not invent facts that are not supported by the review.
    """

    diagnosis = model.invoke(prompt).content

    return {"diagnosis": diagnosis}

In [ ]:
def negative_response(state: ReviewState):
    review = state["review"]
    diagnosis = state["diagnosis"]

    prompt = f"""
    Write a professional and empathetic response
    to this unhappy customer.

    Customer review:
    {review}

    Problem analysis:
    {diagnosis}

    Acknowledge the customer's concern.
    Apologize where appropriate.
    Address the problem and suggest a reasonable next step.
    Do not promise actions that have not been confirmed.
    """

    response = model.invoke(prompt).content

    return {"response": response}

In [ ]:
def positive_response(state: ReviewState):
    review = state["review"]

    prompt = f"""
    Write a warm and professional response
    to this positive customer review.

    Customer review:
    {review}

    Thank the customer, acknowledge their positive feedback,
    and encourage them to continue using the service.
    Keep the response concise and natural.
    """

    response = model.invoke(prompt).content

    return {"response": response}

In [ ]:
def route_review(
    state: ReviewState
) -> Literal["run_diagnosis", "positive_response"]:

    if state["sentiment"] == "negative":
        return "run_diagnosis"

    return "positive_response"

In [ ]:

graph = StateGraph(ReviewState)

# Add nodes
graph.add_node("find_sentiment", find_sentiment)
graph.add_node("run_diagnosis", run_diagnosis)
graph.add_node("negative_response", negative_response)
graph.add_node("positive_response", positive_response)

# Entry point
graph.add_edge(START, "find_sentiment")

# Conditional routing
graph.add_conditional_edges(
    "find_sentiment",
    route_review,
    {
        "run_diagnosis": "run_diagnosis",
        "positive_response": "positive_response",
    },
)

# Negative path
graph.add_edge("run_diagnosis", "negative_response")
graph.add_edge("negative_response", END)

# Positive path
graph.add_edge("positive_response", END)

# Compile the graph
workflow = graph.compile()


In [ ]:
initial_state = {
    "review": (
        "The delivery was three days late, and customer support "
        "did not respond to my messages. Very disappointing."
    ),
    "sentiment": "",
    "diagnosis": "",
    "response": "",
}

output = workflow.invoke(initial_state)

print("SENTIMENT:")
print(output["sentiment"])

print("\nDIAGNOSIS:")
print(output["diagnosis"])

print("\nCUSTOMER RESPONSE:")
print(output["response"])